In [ ]:
import requests
import pandas as pd
import time

# 1. Define variables and years
years = [2019, 2020, 2021, 2022, 2023]
all_census_data = []

# Census Variables: Total Pop, Median Income, Poverty Count
variables = "B01003_001E,B19013_001E,B17001_002E"

print("Starting Multi-Year Census Extraction (2019-2023)...")

for year in years:
    print(f"Fetching data for {year}...")

    # Construct API URL for the specific year
    url = f"https://api.census.gov/data/{year}/acs/acs5?get=NAME,{variables}&for=county:*"

    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            # Convert to DataFrame (skipping the header row from the API)
            df_year = pd.DataFrame(data[1:], columns=data[0])

            # Add the CRITICAL Year column
            df_year['Year'] = year

            all_census_data.append(df_year)
            time.sleep(1) # Be kind to the API
        else:
            print(f"Error for {year}: {response.status_code}")

    except Exception as e:
        print(f"Connection failed for {year}: {e}")

# 2. Combine and Clean the Dataset
if all_census_data:
    census_master = pd.concat(all_census_data, ignore_index=True)

    # Rename columns for clarity
    census_master.rename(columns={
        'B01003_001E': 'total_population',
        'B19013_001E': 'median_income',
        'B17001_002E': 'poverty_count',
        'NAME': 'full_name'
    }, inplace=True)

    # 3. Process Geography
    # Split "Autauga County, Alabama" into separate columns
    census_master[['county', 'state']] = census_master['full_name'].str.split(', ', expand=True)

    # 4. Calculate Poverty Rate
    # Convert strings to numeric first
    numeric_cols = ['total_population', 'median_income', 'poverty_count']
    census_master[numeric_cols] = census_master[numeric_cols].apply(pd.to_numeric, errors='coerce')

    census_master['poverty_rate'] = (census_master['poverty_count'] / census_master['total_population']) * 100

    # 5. Save the multi-year file
    census_master.to_csv('data/acs_county_social_economic.csv', index=False)

    print(f"\nSUCCESS!")
    print(f"Total Census Rows: {len(census_master)}")
    print(f"Years Captured: {census_master['Year'].unique()}")
    print(f"Sample of data:\n{census_master[['Year', 'state', 'county', 'poverty_rate']].head()}")

Starting Multi-Year Census Extraction (2019-2023)...
Fetching data for 2019...
Fetching data for 2020...
Fetching data for 2021...
Fetching data for 2022...
Fetching data for 2023...

✨ SUCCESS!
Total Census Rows: 16106
Years Captured: [2019 2020 2021 2022 2023]
Sample of data:
   Year     state          county  poverty_rate
0  2019  Illinois  Fayette County     15.863668
1  2019  Illinois    Logan County      8.009516
2  2019  Illinois   Saline County     20.571810
3  2019  Illinois     Lake County      7.737005
4  2019  Illinois   Massac County     16.393558
